# Template 04a: PCA Feature Engineering

**Inputs:** data/02_conditioned.parquet
**Outputs:** data/04a_pca_features.parquet

In [ ]:
config_path = "config/car_coll/v1"

In [ ]:
import matplotlib
matplotlib.use('Agg')
import pandas as pd
import yaml
import os
import sys
from pathlib import Path
from IPython.display import Image, display
sys.path.insert(0, str(Path.cwd() / 'lib'))
from utils import setup_notebook_environment
from pca_utils import apply_pca_groups, create_scree_plot, create_loadings_heatmap, save_pca_summary
print('STAGE 04a: PCA FEATURE ENGINEERING')
project_root = setup_notebook_environment()

In [ ]:
config_file = f'{config_path}/config.yaml'
with open(config_file, 'r') as f:
    cfg = yaml.safe_load(f)
output_base = cfg['paths']['output_base']
pca_cfg = cfg.get('feature_engineering', {}).get('pca', {})
pca_enabled = pca_cfg.get('enabled', False)
print(f'Output: {output_base}')
print(f'PCA Enabled: {pca_enabled}')

In [ ]:
plots_dir = f'{output_base}/plots'
results_dir = f'{output_base}/results'
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

In [ ]:
checkpoint_02 = f'{output_base}/data/02_conditioned.parquet'
data = pd.read_parquet(checkpoint_02)
print(f'Loaded: {data.shape}')
join_key = cfg['data']['join_key']
if join_key not in data.columns:
    raise ValueError(f'Join key not found')
print(f'Join key validated')

In [ ]:
if not pca_enabled:
    print('PCA DISABLED - creating empty output')
    pca_output = data[[join_key]].copy()
    output_file = f'{output_base}/data/04a_pca_features.parquet'
    pca_output.to_parquet(output_file, index=False)
    print(f'Saved: {output_file}')
    raise SystemExit(0)

In [ ]:
pca_results, all_pca_features = apply_pca_groups(data, pca_cfg, config_path)
print(f'Total PCA features: {all_pca_features.shape[1]}')

In [ ]:
print('GENERATING PLOTS')
plot_files = {}
for group in pca_results.keys():
    scree_path = create_scree_plot(pca_results[group], plots_dir, group)
    loadings_path = create_loadings_heatmap(pca_results[group], plots_dir, group)
    plot_files[group] = {'scree': scree_path, 'loadings': loadings_path}
    print(f'Created plots for {group}')

In [ ]:
print('DISPLAYING PLOTS')
for group, paths in plot_files.items():
    print(f'PCA Group: {group}')
    display(Image(filename=paths['scree']))
    display(Image(filename=paths['loadings']))

In [ ]:
pca_output = pd.DataFrame({join_key: data[join_key]})
pca_output = pd.concat([pca_output, all_pca_features], axis=1)
output_file = f'{output_base}/data/04a_pca_features.parquet'
pca_output.to_parquet(output_file, index=False)
print(f'Saved: {output_file}')
print(f'Shape: {pca_output.shape}')

In [ ]:
summary_file = save_pca_summary(pca_results, results_dir)
print(f'Summary saved: {summary_file}')

In [ ]:
print('STAGE 04a: COMPLETE')
print(f'PCA Groups: {len(pca_results)}')
print(f'Total Features: {all_pca_features.shape[1]}')